In [1]:
import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer

In [2]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
from nltk.corpus import stopwords
stopwords_list = stopwords.words('english')
stopwords_list

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [4]:
# Tokenize

def tokenize(text):
    text = text.lower() 
    text = text.replace('?', '')
    text = text.replace("'", '')
    return text.split()

# Vocabulary
vocab = {'<UNK>': 0}

def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    merged_tokens = tokenized_question + tokenized_answer
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

df.apply(build_vocab, axis=1)

# Convert words to numeric index
def text_to_indices(text, vocab):
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text 

# text_to_indices("What is the capital of test?", vocab)

In [5]:
import torch 
from torch.utils.data import Dataset, DataLoader

In [6]:
class QADataSet(Dataset):
    def __init__(self, df, vocab):
        self.df = df 
        self.vocab = vocab 

    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        numerical_ques = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        numerical_ans = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

        return torch.tensor(numerical_ques), torch.tensor(numerical_ans)

In [7]:
dataset = QADataSet(df, vocab)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

print(dataset[0][0])

for question, answer in dataloader:
    print(question, answer)

tensor([1, 2, 3, 4, 5, 6])
tensor([[  1,   2,   3, 180, 181, 182, 183]]) tensor([[184]])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]) tensor([[28]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])
tensor([[ 42, 117, 118,   3, 119,  94, 120]]) tensor([[121]])
tensor([[  1,   2,   3,   4,   5, 236, 237]]) tensor([[238]])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor([[85]])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([[36]])
tensor([[ 42, 200,   2,  14, 201, 202, 203, 204]]) tensor([[205]])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([[91]])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([[179]])
tensor([[  1,  87, 229, 230, 231, 232]]) tensor([[233]])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[ 10, 140,   3, 141, 270,  93, 271,   5,   3, 272]]) tensor([[273]])
tensor([[ 42, 137,   2, 138,  39, 139]]) tensor([[53]])
tensor([[ 42,  86,  87, 241, 242,  19,  39, 243]]) tensor([[244]])
tensor([[ 10,  11, 157, 158, 15

In [8]:
import torch.nn as nn 

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)        

    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))

        return output



In [9]:
x = nn.Embedding(324, embedding_dim=50)

x(dataset[0][0]).shape

torch.Size([6, 50])

In [10]:
learning_rate = 0.001 
epochs = 20

In [11]:
model = SimpleRNN(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [12]:
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader: 
        optimizer.zero_grad()
        # Forward pass
        output = model(question)
        # Loss calculation
        loss = criterion(output, answer[0])
        # Gradient
        loss.backward()
        # update
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss {total_loss}")

Epoch 1: Loss 521.3627319335938
Epoch 2: Loss 451.2505717277527
Epoch 3: Loss 372.3491380214691
Epoch 4: Loss 314.50212574005127
Epoch 5: Loss 264.02592277526855
Epoch 6: Loss 216.0998980998993
Epoch 7: Loss 171.44712018966675
Epoch 8: Loss 133.3395470380783
Epoch 9: Loss 102.66703775525093
Epoch 10: Loss 78.28399667143822
Epoch 11: Loss 60.51155659556389
Epoch 12: Loss 47.36506040394306
Epoch 13: Loss 37.836022421717644
Epoch 14: Loss 30.900480166077614
Epoch 15: Loss 25.35226282477379
Epoch 16: Loss 21.241214513778687
Epoch 17: Loss 17.971385687589645
Epoch 18: Loss 15.354985490441322
Epoch 19: Loss 13.148588068783283
Epoch 20: Loss 11.456063508987427


In [18]:
def predict(model, question, threshold):
    numerical_question = text_to_indices(question, vocab)
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)
    output = model(question_tensor)
    probs = torch.nn.functional.softmax(output, dim=1)
    print(output.shape)
    value, index = torch.max(probs, dim=1)
    print(value, index)
    if value < threshold:
        print("I don't know")
    else: 
        print(list(vocab.keys())[index])

predict(model, "What is the capital of France?", 0.5)



torch.Size([1, 324])
tensor([0.9078], grad_fn=<MaxBackward0>) tensor([7])
paris
